# Producer Price Validation: Does the Risk Index Predict Real Price Shocks?

**Purpose:**  

> *"If the index measures something real, high-scoring countries should show  
> larger price responses around 2008, 2011 and 2022. That is a genuine  
> out-of-sample check rather than another correlation with our own inputs."*

The logic: producer prices are **not an input** to the risk index (the index  
uses quantity-based HHI, IDR, and partner count — no price data). So if  
high-risk countries also experienced larger price spikes during known food  
crises, it constitutes genuine external validation.

---

## Important Caveat on Producer Prices

FAOSTAT producer prices are **farm-gate prices in the producing country**,  
not import prices. They describe an exporter's economics, not what an importer  
pays. However, global commodity prices transmit across borders, so a country  
with high import dependency *should* show producer price volatility tracking  
global shocks.

If `producer_prices_cleaned.csv` is not available (it requires the raw FAOSTAT  
download), this notebook falls back to **import unit values** (value/quantity)  
computed from the trade matrix as a proxy.

---

## Known Crisis Years

| Year | Event |
|---|---|
| 2007-2008 | Global food price crisis (wheat, rice, maize) |
| 2010-2011 | Drought-driven spike (Russia wheat export ban 2010) |
| 2022 | Black Sea disruption (Russia-Ukraine war) |
| 2023 | India rice export ban |

---

## 0. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy import stats
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)

# ---- Paths ----
ROOT = Path(".").resolve().parent
TRADE_MATRIX = ROOT / "data" / "cleaned" / "trade_matrix_cleaned.csv"
PRICES_PATH = ROOT / "data" / "cleaned" / "producer_prices_cleaned.csv"
CONC_PATH = ROOT / "data" / "cleaned" / "concentration_with_shannon.csv"
FBS_PATH = ROOT / "data" / "cleaned" / "fbs_cleaned.csv"
VIZ_DIR = ROOT / "visualizations"
OUTPUT_DIR = ROOT / "data" / "cleaned"
VIZ_DIR.mkdir(parents=True, exist_ok=True)

# ---- Item mapping ----
ITEM_MAP = {
    "Wheat": "Wheat",
    "Wheat and meslin flour": "Wheat",
    "Rice, paddy (rice milled equivalent)": "Rice",
    "Rice, milled": "Rice",
    "Maize (corn)": "Maize",
}

COMMODITIES = ["Wheat", "Rice", "Maize"]

# ---- Crisis windows ----
# Each crisis is defined as a (peak_year, pre_year) pair.
# Price spike = (price at peak - price at pre) / price at pre.
CRISES = [
    {"name": "2007-08 Food Crisis", "peak": 2008, "pre": 2006},
    {"name": "2010-11 Drought Spike", "peak": 2011, "pre": 2009},
    {"name": "2022 Black Sea", "peak": 2022, "pre": 2020},
]

print(f"Producer prices available: {PRICES_PATH.exists()}")
print(f"Concentration CSV available: {CONC_PATH.exists()}")
print(f"FBS available: {FBS_PATH.exists()}")
print(f"Trade matrix available: {TRADE_MATRIX.exists()}")

## 1. Load Price Data

Try producer prices first. If not available, compute import unit values  
from the trade matrix (import value / import quantity = USD per tonne).

In [ ]:
USE_PRODUCER_PRICES = PRICES_PATH.exists()

if USE_PRODUCER_PRICES:
    # ---- Load producer prices ----
    print("Loading producer prices...")
    pp = pd.read_csv(PRICES_PATH)
    
    # Filter to USD/tonne producer price for our staple items
    pp_usd = pp[
        pp["Element"].str.contains("Producer Price", case=False, na=False)
        & pp["Unit"].str.contains("USD", case=False, na=False)
    ].copy()
    
    # Map items to commodities
    # Producer prices use slightly different item names; map flexibly
    pp_item_map = {}
    for item in pp_usd["Item"].unique():
        item_lower = item.lower()
        if "wheat" in item_lower:
            pp_item_map[item] = "Wheat"
        elif "rice" in item_lower:
            pp_item_map[item] = "Rice"
        elif "maize" in item_lower or "corn" in item_lower:
            pp_item_map[item] = "Maize"
    
    pp_usd["Commodity"] = pp_usd["Item"].map(pp_item_map)
    pp_usd = pp_usd[pp_usd["Commodity"].notna()].copy()
    pp_usd["Value"] = pd.to_numeric(pp_usd["Value"], errors="coerce")
    
    prices = (
        pp_usd.groupby(
            ["Area Code", "Area", "Year", "Commodity"],
            as_index=False,
        )["Value"].mean()
        .rename(columns={
            "Area Code": "Reporter Country Code",
            "Area": "Reporter Countries",
            "Value": "price_usd_per_t",
        })
    )
    price_label = "Producer Price (USD/t)"
    print(f"Producer price observations: {len(prices):,}")
    
else:
    # ---- Fallback: import unit values from trade matrix ----
    # Unit value = (import value in 1000 USD * 1000) / import quantity in tonnes
    print("Producer prices not available. Computing import unit values...")
    print("NOTE: Unit values are noisy (aggregation artifacts, reporting delays).")
    
    raw = pd.read_csv(TRADE_MATRIX)
    
    # Get import value rows
    val_rows = raw[
        (raw["Element"] == "Import value")
        & (raw["Value"] > 0)
    ].copy()
    val_rows["Commodity"] = val_rows["Item"].map(ITEM_MAP)
    val_rows = val_rows[val_rows["Commodity"].notna()].copy()
    
    imp_val = (
        val_rows.groupby(
            ["Reporter Country Code", "Reporter Countries", "Commodity", "Year"],
            as_index=False,
        )["Value"].sum()
        .rename(columns={"Value": "import_value_1000usd"})
    )
    
    # Get import quantity rows
    qty_rows = raw[
        (raw["Element"] == "Import quantity")
        & (raw["Unit"] == "t")
        & (raw["Value"] > 0)
    ].copy()
    qty_rows["Commodity"] = qty_rows["Item"].map(ITEM_MAP)
    qty_rows = qty_rows[qty_rows["Commodity"].notna()].copy()
    
    imp_qty = (
        qty_rows.groupby(
            ["Reporter Country Code", "Reporter Countries", "Commodity", "Year"],
            as_index=False,
        )["Value"].sum()
        .rename(columns={"Value": "import_quantity_t"})
    )
    
    # Compute unit value
    prices = imp_val.merge(
        imp_qty,
        on=["Reporter Country Code", "Reporter Countries", "Commodity", "Year"],
        how="inner",
    )
    prices["price_usd_per_t"] = (
        prices["import_value_1000usd"] * 1000 / prices["import_quantity_t"]
    )
    
    # Drop extreme outliers (unit value artifacts)
    for commodity in COMMODITIES:
        mask = prices["Commodity"] == commodity
        p01 = prices.loc[mask, "price_usd_per_t"].quantile(0.01)
        p99 = prices.loc[mask, "price_usd_per_t"].quantile(0.99)
        prices.loc[mask, "price_usd_per_t"] = prices.loc[mask, "price_usd_per_t"].clip(
            lower=p01, upper=p99
        )
    
    prices = prices[
        ["Reporter Country Code", "Reporter Countries", "Commodity", "Year", "price_usd_per_t"]
    ]
    price_label = "Import Unit Value (USD/t)"
    print(f"Import unit value observations: {len(prices):,}")

print(f"Price metric: {price_label}")
prices.head()

## 2. Compute Price Spikes Around Crisis Years

For each crisis, compute the **relative price change** per country-commodity:  
`spike = (price_peak - price_pre) / price_pre`

A spike of 0.5 means price rose 50% from pre-crisis to peak.

In [ ]:
spike_records = []

for crisis in CRISES:
    name = crisis["name"]
    peak_year = crisis["peak"]
    pre_year = crisis["pre"]
    
    # Get prices at peak and pre year
    p_peak = prices[prices["Year"] == peak_year][
        ["Reporter Country Code", "Reporter Countries", "Commodity", "price_usd_per_t"]
    ].rename(columns={"price_usd_per_t": "price_peak"})
    
    p_pre = prices[prices["Year"] == pre_year][
        ["Reporter Country Code", "Reporter Countries", "Commodity", "price_usd_per_t"]
    ].rename(columns={"price_usd_per_t": "price_pre"})
    
    merged = p_peak.merge(
        p_pre,
        on=["Reporter Country Code", "Reporter Countries", "Commodity"],
        how="inner",
    )
    
    # Compute relative spike (avoid division by zero)
    merged["price_spike"] = (
        (merged["price_peak"] - merged["price_pre"])
        / merged["price_pre"].replace(0, np.nan)
    )
    merged["crisis"] = name
    merged["peak_year"] = peak_year
    merged["pre_year"] = pre_year
    
    spike_records.append(merged)
    
    n = len(merged)
    med = merged["price_spike"].median()
    print(f"{name}: {n} country-commodity observations, median spike = {med:+.1%}")

spikes = pd.concat(spike_records, ignore_index=True)
print(f"\nTotal spike observations: {len(spikes):,}")

## 3. Compute Pre-Crisis Risk Scores

The risk score must be computed from data **before** the crisis (otherwise it's  
circular). For each crisis, we use the concentration metrics from the year  
before the pre-crisis year.

In [ ]:
# ---- Load or compute concentration ----
if CONC_PATH.exists():
    conc = pd.read_csv(CONC_PATH)
else:
    # Compute inline (same logic as other notebooks)
    if "raw" not in dir():
        raw = pd.read_csv(TRADE_MATRIX)
    imp = raw[
        (raw["Element"] == "Import quantity")
        & (raw["Unit"] == "t")
        & (raw["Value"] > 0)
    ].copy()
    imp["Commodity"] = imp["Item"].map(ITEM_MAP)
    imp = imp[imp["Commodity"].notna()].copy()
    GK = ["Reporter Country Code", "Reporter Countries", "Commodity", "Year"]
    FK = GK + ["Partner Country Code", "Partner Countries"]
    flows = imp.groupby(FK, as_index=False)["Value"].sum().rename(columns={"Value": "t"})
    tot = flows.groupby(GK, as_index=False)["t"].sum().rename(columns={"t": "total"})
    flows = flows.merge(tot, on=GK, how="left")
    flows["share"] = flows["t"] / flows["total"]
    conc = flows.groupby(GK, as_index=False).agg(
        partner_hhi=("share", lambda x: (x ** 2).sum()),
        partner_count=("Partner Country Code", "nunique"),
        top_partner_share=("share", "max"),
        total_import_quantity_t=("t", "sum"),
    )

# ---- For each crisis, get the pre-crisis concentration ----
# Use the year BEFORE the pre-crisis year to avoid look-ahead.
risk_records = []

for crisis in CRISES:
    risk_year = crisis["pre"] - 1  # e.g. for 2008 crisis, use 2005 risk
    
    # Average over a 3-year window ending at risk_year for stability
    risk_window = conc[conc["Year"].between(risk_year - 2, risk_year)].copy()
    
    if risk_window.empty:
        print(f"WARNING: No concentration data for {crisis['name']} "
              f"(risk window {risk_year-2}-{risk_year})")
        continue
    
    risk_avg = risk_window.groupby(
        ["Reporter Country Code", "Reporter Countries", "Commodity"],
        as_index=False,
    ).agg(
        pre_crisis_hhi=("partner_hhi", "mean"),
        pre_crisis_partner_count=("partner_count", "mean"),
        pre_crisis_top_share=("top_partner_share", "mean"),
    )
    risk_avg["crisis"] = crisis["name"]
    risk_records.append(risk_avg)
    
    print(f"{crisis['name']}: risk from {risk_year-2}-{risk_year}, "
          f"{len(risk_avg)} country-commodity groups")

pre_crisis_risk = pd.concat(risk_records, ignore_index=True)

## 4. Join Risk Scores with Price Spikes

The core test: do countries with higher pre-crisis concentration experience  
larger price spikes during the crisis?

In [ ]:
# ---- Merge spikes with pre-crisis risk ----
validation = spikes.merge(
    pre_crisis_risk,
    on=["Reporter Country Code", "Reporter Countries", "Commodity", "crisis"],
    how="inner",
)

# Drop NaN spikes (missing price data)
validation = validation.dropna(subset=["price_spike", "pre_crisis_hhi"]).copy()

print(f"Validation dataset: {len(validation):,} observations")
print(f"Crises covered: {validation['crisis'].unique().tolist()}")
print(f"Commodities: {validation['Commodity'].unique().tolist()}")

## 5. Correlation: Pre-Crisis HHI vs Price Spike

In [ ]:
# ---- Overall correlation ----
r_overall, p_overall = stats.pearsonr(
    validation["pre_crisis_hhi"], validation["price_spike"]
)
r_spearman, p_spearman = stats.spearmanr(
    validation["pre_crisis_hhi"], validation["price_spike"]
)

print("OVERALL CORRELATION: Pre-Crisis HHI vs Price Spike")
print("=" * 55)
print(f"  Pearson r  = {r_overall:.4f}  (p = {p_overall:.2e})")
print(f"  Spearman r = {r_spearman:.4f}  (p = {p_spearman:.2e})")

if r_overall > 0 and p_overall < 0.05:
    print("\n  --> POSITIVE and significant: higher concentration predicts")
    print("      larger price spikes. The index measures something real.")
elif r_overall > 0:
    print("\n  --> Positive but not significant at p<0.05.")
    print("      Direction is consistent but evidence is weak.")
else:
    print("\n  --> Not positive. The relationship may not hold for")
    print("      producer prices / import unit values.")

In [ ]:
# ---- Per-crisis, per-commodity correlations ----
corr_results = []

for crisis_name in validation["crisis"].unique():
    for commodity in validation["Commodity"].unique():
        sub = validation[
            (validation["crisis"] == crisis_name)
            & (validation["Commodity"] == commodity)
        ]
        if len(sub) < 10:
            continue
        
        r, p = stats.pearsonr(sub["pre_crisis_hhi"], sub["price_spike"])
        corr_results.append({
            "Crisis": crisis_name,
            "Commodity": commodity,
            "N": len(sub),
            "Pearson_r": r,
            "p_value": p,
            "Median_spike": sub["price_spike"].median(),
        })

corr_df = pd.DataFrame(corr_results)
print("\nPer-crisis, per-commodity correlations (HHI vs price spike):")
print(corr_df.to_string(index=False, float_format="{:.3f}".format))

## 6. Visualization: HHI vs Price Spike by Crisis

In [ ]:
crises_in_data = validation["crisis"].unique()
n_crises = len(crises_in_data)

if n_crises > 0:
    fig, axes = plt.subplots(1, n_crises, figsize=(7 * n_crises, 6), squeeze=False)
    axes = axes[0]  # flatten
    
    for ax, crisis_name in zip(axes, crises_in_data):
        sub = validation[validation["crisis"] == crisis_name]
        
        for commodity in COMMODITIES:
            cs = sub[sub["Commodity"] == commodity]
            if not cs.empty:
                ax.scatter(
                    cs["pre_crisis_hhi"], cs["price_spike"] * 100,
                    alpha=0.4, s=25, label=commodity, edgecolors="none",
                )
        
        # Trend line (all commodities pooled)
        if len(sub) > 5:
            z = np.polyfit(sub["pre_crisis_hhi"], sub["price_spike"] * 100, 1)
            x_line = np.linspace(sub["pre_crisis_hhi"].min(), sub["pre_crisis_hhi"].max(), 50)
            ax.plot(x_line, np.polyval(z, x_line), "--", color="grey", linewidth=1.5)
        
        r = sub[["pre_crisis_hhi", "price_spike"]].corr().iloc[0, 1]
        ax.annotate(f"r = {r:.3f}", xy=(0.05, 0.95), xycoords="axes fraction",
                    fontsize=11, va="top")
        
        ax.set_xlabel("Pre-crisis HHI (supplier concentration)")
        ax.set_ylabel(f"Price spike (%)")
        ax.set_title(crisis_name, fontsize=12, weight="bold")
        ax.legend(fontsize=9)
        ax.axhline(0, color="grey", linewidth=0.5, alpha=0.5)
    
    fig.suptitle(
        f"Pre-Crisis Supplier Concentration vs {price_label} Spike\n"
        "Positive slope = concentration predicts larger spikes (validates the risk index)",
        fontsize=13, y=1.05,
    )
    plt.tight_layout()
    plt.savefig(str(VIZ_DIR / "price_validation_hhi_vs_spike.png"),
                dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("No crisis data available for plotting.")

## 7. Grouped Comparison: High-Risk vs Low-Risk Countries

Split countries into high-concentration (above median HHI) and low-concentration  
(below median) groups. Do high-concentration countries show statistically  
larger price spikes?

In [ ]:
# ---- Split into high/low concentration groups per crisis-commodity ----
group_results = []

for crisis_name in validation["crisis"].unique():
    for commodity in validation["Commodity"].unique():
        sub = validation[
            (validation["crisis"] == crisis_name)
            & (validation["Commodity"] == commodity)
        ].copy()
        
        if len(sub) < 10:
            continue
        
        median_hhi = sub["pre_crisis_hhi"].median()
        sub["risk_group"] = np.where(
            sub["pre_crisis_hhi"] >= median_hhi, "High HHI", "Low HHI"
        )
        
        high = sub[sub["risk_group"] == "High HHI"]["price_spike"]
        low = sub[sub["risk_group"] == "Low HHI"]["price_spike"]
        
        # Mann-Whitney U test (non-parametric, no normality assumption)
        if len(high) >= 5 and len(low) >= 5:
            u_stat, u_p = stats.mannwhitneyu(high, low, alternative="greater")
        else:
            u_stat, u_p = np.nan, np.nan
        
        group_results.append({
            "Crisis": crisis_name,
            "Commodity": commodity,
            "N_high": len(high),
            "N_low": len(low),
            "Median_spike_high": high.median(),
            "Median_spike_low": low.median(),
            "Diff": high.median() - low.median(),
            "MannWhitney_p": u_p,
        })

group_df = pd.DataFrame(group_results)

print("HIGH-CONCENTRATION vs LOW-CONCENTRATION: Median Price Spikes")
print("=" * 70)
print("(Positive Diff = high-conc countries had larger spikes)")
print(
    group_df.to_string(
        index=False,
        float_format="{:.3f}".format,
        formatters={"MannWhitney_p": "{:.3e}".format},
    )
)

In [ ]:
# ---- Box plots: high vs low concentration by crisis ----
if len(validation) > 0:
    validation["risk_group"] = np.where(
        validation["pre_crisis_hhi"] >= validation.groupby(
            ["crisis", "Commodity"]
        )["pre_crisis_hhi"].transform("median"),
        "High HHI", "Low HHI",
    )
    
    g = sns.catplot(
        data=validation,
        x="Commodity", y="price_spike",
        hue="risk_group",
        col="crisis",
        kind="box",
        height=5, aspect=1.2,
        palette={"High HHI": "#d32f2f", "Low HHI": "#1565c0"},
        showfliers=False,
    )
    
    g.set_axis_labels("Commodity", "Price Spike (ratio)")
    g.set_titles("{col_name}")
    g.figure.suptitle(
        "Price Spikes: High vs Low Pre-Crisis Supplier Concentration",
        fontsize=14, y=1.03,
    )
    plt.tight_layout()
    plt.savefig(str(VIZ_DIR / "price_validation_boxplot.png"),
                dpi=150, bbox_inches="tight")
    plt.show()

## 8. Summary & Validation Verdict

In [ ]:
print("=" * 60)
print("PRODUCER PRICE VALIDATION: SUMMARY")
print("=" * 60)

print(f"\nPrice metric used: {price_label}")
print(f"Total observations: {len(validation):,}")
print(f"Crises tested: {validation['crisis'].nunique()}")

print(f"\nOverall correlation (HHI vs spike):")
print(f"  Pearson r  = {r_overall:+.4f}  (p = {p_overall:.2e})")
print(f"  Spearman r = {r_spearman:+.4f}  (p = {p_spearman:.2e})")

# Count how many crisis-commodity pairs show positive correlation
if len(corr_df) > 0:
    n_positive = (corr_df["Pearson_r"] > 0).sum()
    n_sig_positive = ((corr_df["Pearson_r"] > 0) & (corr_df["p_value"] < 0.05)).sum()
    n_total = len(corr_df)
    print(f"\nPer crisis-commodity pairs:")
    print(f"  {n_positive}/{n_total} show positive r (concentration -> larger spike)")
    print(f"  {n_sig_positive}/{n_total} are statistically significant (p<0.05)")

if len(group_df) > 0:
    n_higher = (group_df["Diff"] > 0).sum()
    n_sig_higher = ((group_df["Diff"] > 0) & (group_df["MannWhitney_p"] < 0.05)).sum()
    print(f"\nGroup comparison (high vs low HHI):")
    print(f"  {n_higher}/{len(group_df)} crisis-commodity pairs: high-HHI group")
    print(f"  had larger median spike")
    print(f"  {n_sig_higher}/{len(group_df)} significant (Mann-Whitney p<0.05)")

print("\nVERDICT:")
if r_overall > 0 and p_overall < 0.05:
    print("  The risk index receives external validation: countries with")
    print("  higher pre-crisis supplier concentration experienced")
    print("  significantly larger price responses during food crises.")
elif r_overall > 0:
    print("  The direction is consistent (positive r) but not statistically")
    print("  significant. This is expected given producer prices measure")
    print("  farm-gate, not import, prices. The result neither confirms")
    print("  nor contradicts the index.")
else:
    print("  No positive relationship found. This may reflect the gap")
    print("  between producer prices (farm-gate) and the import price")
    print(f"  channel the risk index measures. Consider using import")
    print(f"  unit values or CPI food index for a stronger test.")

print("\nVisualizations saved:")
print("  - price_validation_hhi_vs_spike.png")
print("  - price_validation_boxplot.png")